# Workshop 4 — Fase 4: localizar el deterioro del AE

## Plan y controles antes de ejecutar

1. Montar Drive, verificar los cuatro logs y ambos checkpoints históricos.
2. Crear copias estáticas de logs en una carpeta nueva; comprobar hashes y
   preparar su ZIP descargable. No usar `/content/audit_ae_z32/` como persistencia.
3. Auditar épocas 8–15 por sesión y cobertura; distinguir valores ausentes de
   cero eventos. Separar validación completa de los 128 ejemplos diagnósticos.
4. Evaluar checkpoints en lectura si se desea ampliar los diagnósticos.
5. Revisar el plan de una ventana de hasta 64 updates del **AE original z32**,
   con evaluación/checkpoint cada 8. No lanzar entrenamiento desde este panel
   sin activar `RUN_WINDOW`. No se llama a la matriz de 12 experimentos.

Controles: mismo prefijo y orden de train, mismo val de 128, CE contra x,
baseline ajustado solo con todo train, swap sin puntos fijos, FP64 para normas,
clipping y métricas; Adam/pesos/estados finitos después de cada paso. Guardar
mejor estado sano del diagnóstico y checkpoints periódicos inmutables.

Parar inmediatamente por fallo numérico, norma raw >1e8, |z|>100, o CE ≥2 veces
su mejor referencia; parar tras dos evaluaciones con CE ≥1.2 veces la mejor o
daño swap ≤50% de su referencia previamente informativa (≥1 nat/secuencia).
Clipping al 100% por sí solo no obliga a parar. Los umbrales son controles
prespecificados, no una causa raíz demostrada ni una calibración óptima.

**RUN_WINDOW=False por defecto.** Límite duro de 256 updates acumulados por
línea diagnóstica y 600 s por invocación. Esta instrumentación corta no alcanza
por sí sola los millones de ejemplos y miles de updates del episodio histórico.
No se cambia la arquitectura ni se inicia AE z64/VAE.


In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
from pathlib import Path
from datetime import datetime, timezone
import json, subprocess, sys, zipfile
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")


In [ ]:
# @title Rutas confirmadas y paquete NUEVO de fase 4
HISTORICAL_RUN = "/content/drive/MyDrive/Workshop_4/runs/workshop4_clipping_v2_prueba12/ae_z32" # @param {type:"string"}
ZIP_SOURCE = "Upload" # @param ["Upload", "Drive"]
PROJECT_ZIP = "/content/drive/MyDrive/Workshop_4/workshop4_colab_deterioration_phase4.zip" # @param {type:"string"}
EXTRACT_DIR = "/content/workshop4_deterioration_phase4" # @param {type:"string"}
PROCESSED_DIR = "/content/drive/MyDrive/Workshop_4/data/processed" # @param {type:"string"}
OUTPUT_PARENT = "/content/drive/MyDrive/Workshop_4/deterioration_phase4" # @param {type:"string"}
historical = Path(HISTORICAL_RUN)
source_logs = {"history": historical / "history.csv",
               "gradient_norms": historical / "gradient_norms.csv",
               "events": historical / "nonfinite_events.jsonl",
               "metadata": historical / "metadata.json"}
BEST_CHECKPOINT = historical / "checkpoints/best.pt"  # época 8, lectura
LAST_CHECKPOINT = historical / "checkpoints/last.pt"  # época 30, lectura
presence = {name: p.is_file() for name, p in source_logs.items()}
presence.update(best=BEST_CHECKPOINT.is_file(), last=LAST_CHECKPOINT.is_file())
print(json.dumps(presence, indent=2))
missing = [str(p) for p in [*source_logs.values(), BEST_CHECKPOINT, LAST_CHECKPOINT] if not p.is_file()]
if missing:
    print("Archivos pendientes de verificar/corregir:", missing)
print("No se usan copias efímeras de /content/audit_ae_z32/")


In [ ]:
if ZIP_SOURCE == "Upload":
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Sube exactamente el ZIP nuevo de fase 4")
    PROJECT_ZIP = next(iter(uploaded))
zip_path = Path(PROJECT_ZIP).expanduser().resolve()
with zipfile.ZipFile(zip_path) as archive:
    helpers = [n for n in archive.namelist() if n.endswith("Workshop_4/src/colab_utils.py")]
    if len(helpers) != 1:
        raise ValueError("ZIP de proyecto incompatible")
    zip_import_root = helpers[0].removesuffix("src/colab_utils.py")
sys.path.insert(0, str(zip_path) + "/" + zip_import_root)
from src.colab_utils import extract_project_zip, runtime_info
WORKSHOP_DIR = extract_project_zip(zip_path, EXTRACT_DIR)
assert (WORKSHOP_DIR / "src/deterioration_control.py").is_file()
from src.deterioration_audit import stage_log_bundle
print("Código:", WORKSHOP_DIR)
print(runtime_info())


## Copias estáticas y ZIP de los logs

La siguiente celda copia exclusivamente history.csv, gradient_norms.csv,
nonfinite_events.jsonl y metadata.json a una carpeta nueva. Verifica hashes
antes/después y el contenido del ZIP; no incluye pesos ni escribe en el run
histórico. Si una fuente cambia o falta, no se declara un bundle válido.
Se puede descargar el ZIP para analizarlo después en el repositorio local.


In [ ]:
# @title Preparar y, opcionalmente, descargar los cuatro logs
PREPARE_LOG_COPIES = True # @param {type:"boolean"}
DOWNLOAD_LOG_ZIP = False # @param {type:"boolean"}
bundle = None
if PREPARE_LOG_COPIES:
    if not all(p.is_file() for p in source_logs.values()):
        print("Logs incompletos; corrige las rutas. No se inventan eventos ni valores.")
    else:
        copied_dir = Path("/content/workshop4_phase4_inputs") / f"logs_{run_id}"
        bundle = stage_log_bundle(historical, copied_dir,
                                  Path(OUTPUT_PARENT) / f"logs_{run_id}.zip")
        print(bundle["status"], bundle["zip_path"], bundle["zip_sha256"])
        if DOWNLOAD_LOG_ZIP:
            files.download(bundle["zip_path"])


In [ ]:
# Auditoría de SOLO LECTURA: primero muestra el plan; después analiza las copias.
RUN_LOG_AUDIT = True # @param {type:"boolean"}
AUDIT_DIR = Path(OUTPUT_PARENT) / f"audit_{run_id}"
if bundle is not None:
    paths = bundle["copied_files"]
    audit_command = [sys.executable, "-m", "src.deterioration_audit",
                     "--output-dir", str(AUDIT_DIR), "--start-epoch", "8", "--end-epoch", "15"]
    for flag, key in (("--history", "history"), ("--gradient-norms", "gradient_norms"),
                      ("--events", "events"), ("--metadata", "metadata")):
        audit_command += [flag, paths[key]]
    subprocess.run(audit_command, cwd=WORKSHOP_DIR, check=True, timeout=30)
    if RUN_LOG_AUDIT:
        subprocess.run(audit_command + ["--execute"], cwd=WORKSHOP_DIR, check=True, timeout=300)
        audit = json.loads((AUDIT_DIR / "report.json").read_text())
        print("Causa raíz establecida:", audit["root_cause_established"])
        print("Informe:", AUDIT_DIR / "report.json")
else:
    print("Auditoría empírica de producción pendiente de copias estáticas válidas.")


## Evaluación opcional de checkpoints históricos

La evaluación sobre 128 y los probes sobre 8 ejemplos se guardan por separado.
Nunca restauran Adam ni modifican best.pt/last.pt. Los probes de estados,
gradientes por bloque y VJP usan el backend de referencia y no son mediciones
de throughput. El baseline recorre TODO train. La existencia de momentos Adam
en un best de inferencia no demuestra que correspondan a sus pesos.


In [ ]:
EVALUATE_HISTORICAL = False # @param {type:"boolean"}
DEVICE = "cuda" # @param ["cuda", "cpu"]
if EVALUATE_HISTORICAL:
    if not BEST_CHECKPOINT.is_file() or not LAST_CHECKPOINT.is_file():
        raise FileNotFoundError("Faltan checkpoints históricos; verifica las rutas después de montar Drive")
    checkpoint_dir = Path(OUTPUT_PARENT) / f"checkpoint_eval_{run_id}"
    cmd = [sys.executable, "-m", "src.latent_diagnostic_runner",
           "--processed-dir", str(PROCESSED_DIR), "--device", DEVICE,
           "--checkpoint", "best=" + str(BEST_CHECKPOINT),
           "--checkpoint", "last=" + str(LAST_CHECKPOINT),
           "--output-dir", str(checkpoint_dir)]
    subprocess.run(cmd, cwd=WORKSHOP_DIR, check=True, timeout=30)
    subprocess.run(cmd + ["--execute"], cwd=WORKSHOP_DIR, check=True, timeout=650)
    probes = [sys.executable, "-m", "src.audit_model", "--best", str(BEST_CHECKPOINT),
              "--last", str(LAST_CHECKPOINT), "--processed-dir", str(PROCESSED_DIR),
              "--device", DEVICE, "--batch-size", "8", "--output", str(Path(OUTPUT_PARENT) / f"structural_probes_{run_id}.json")]
    subprocess.run(probes, cwd=WORKSHOP_DIR, check=True, timeout=180)


## Ventana controlada — revisión obligatoria del plan

`weights_only` carga una copia lógica de los pesos de best8 y crea Adam nuevo:
es una rama diagnóstica explícita, **no una reanudación del entrenamiento
histórico ni un experimento causal aislado sobre Adam**. Cambian también el
cursor/orden de datos respecto al histórico. `continue` solo acepta checkpoints
periódicos nuevos de fase 4 y conserva Adam, RNG, cursor, datos y controles.
`reset_adam` acepta esos mismos checkpoints nuevos y conserva pesos/cursor/RNG/
orden/controles con Adam nuevo, para compararlo con `continue` sobre los mismos
lotes siguientes. Cada rama necesita salida nueva y activación explícita.

`fresh` sirve para comprobar instrumentación. Ninguno de esos modos reproduce
por sí solo la historia de épocas 8–15. Las etiquetas de la ventana son updates,
no épocas reconstruidas. Para localizar el fallo histórico hacen falta logs
densos o un estado completo sano cercano al inicio del deterioro.


In [ ]:
# @title Panel de ventana corta: desactivada
INITIALIZATION = "weights_only" # @param ["weights_only", "fresh", "continue", "reset_adam"]
SOURCE_CHECKPOINT = str(BEST_CHECKPOINT) # @param {type:"string"}
RUN_WINDOW = False # @param {type:"boolean"}
FULL_VALIDATION = False # @param {type:"boolean"}
WINDOW_DIR = Path(OUTPUT_PARENT) / f"window_{run_id}"
window_command = [sys.executable, "-m", "src.deterioration_control",
                  "--config", str(WORKSHOP_DIR / "config/deterioration_control.json"),
                  "--processed-dir", str(PROCESSED_DIR), "--device", DEVICE,
                  "--initialization", INITIALIZATION, "--output-dir", str(WINDOW_DIR)]
if INITIALIZATION != "fresh":
    window_command += ["--source-checkpoint", SOURCE_CHECKPOINT]
if FULL_VALIDATION:
    window_command.append("--full-validation")
print("Plan y salida nueva:", WINDOW_DIR)
subprocess.run(window_command, cwd=WORKSHOP_DIR, check=True, timeout=30)


In [ ]:
if RUN_WINDOW:
    if INITIALIZATION != "fresh" and not Path(SOURCE_CHECKPOINT).is_file():
        raise FileNotFoundError(SOURCE_CHECKPOINT)
    completed = subprocess.run(window_command + ["--execute"], cwd=WORKSHOP_DIR,
                               check=False, timeout=650)
    print("Código de salida:", completed.returncode)
    if (WINDOW_DIR / "summary.json").is_file():
        summary = json.loads((WINDOW_DIR / "summary.json").read_text())
        print("Estado:", summary["status"], summary.get("reason"))
        print("Mejor diagnóstico:", summary.get("best_prefix"))
    else:
        print("Sin resumen final: corte duro; resultados parciales, no ventana completada.")
else:
    print("Plan revisable; no se ejecutó ninguna actualización de entrenamiento.")


In [ ]:
# Visualización de la ventana, únicamente si se ejecutó.
import matplotlib.pyplot as plt
if RUN_WINDOW and (WINDOW_DIR / "summary.json").is_file():
    summary = json.loads((WINDOW_DIR / "summary.json").read_text())
    snapshots = summary.get("evaluations", [])
    if snapshots:
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        updates = [r["cumulative_update"] for r in snapshots]
        for ax, key, title in zip(axes, ("reconstruction", "swap_damage", "swap_tv"),
                                   ("CE (nats/secuencia)", "Daño swap (nats/secuencia)", "TV original/swap")):
            ax.plot(updates, [r["metrics"][key] for r in snapshots], marker="o")
            ax.set(title=title, xlabel="Updates acumulados de la ventana")
            ax.grid(alpha=.2)
        plt.tight_layout()
        plt.show()


## Lectura de resultados y siguiente decisión

- CE baja con pérdida importante de utilidad swap genera alerta; no basta con
  conservar el mínimo de CE si el decoder deja de aprovechar el ejemplo.
- Los archivos `checkpoints/update_*.pt` son inmutables. `best_prefix.json`
  referencia el mejor estado elegible; `latest_checkpoint.json` indica el más
  reciente. No sustituir el best histórico con el último checkpoint de ventana.
  Conservar la carpeta anterior al continuar: descargar solo un periódico puede
  dejar sin su best referenciado; el runner rechaza esa referencia ausente.
- `FULL_VALIDATION=True` calcula CE completa al inicio y al finalizar una ventana
  completada (se omite la final tras una parada), en otra escala
  muestral; los controles de swap siguen usando solo el prefijo fijo. No se
  promueve un best de prefijo como mejor modelo de validación completa.
- Una parada numérica preserva checkpoints anteriores; un archivo temporal o
  proceso cortado no constituye un checkpoint confirmado. No repetir
  automáticamente ventanas ni bajar umbrales después de ver los datos.

Consultar `AUDIT_REPORT_PHASE4.md`. La meta posterior de anomalías con AE/VAE
requiere, además de estabilidad, entrenamiento normal-only, calibración separada
del umbral y evaluación final sin fuga de test. Esta fase prepara ese paso y no
ejecuta AE z64, VAE ni detección de anomalías automáticamente.
